# Refusal follow-up pipeline

Run the steps in order. All file paths are defined in the next cell. `scripts/code.py` (RAW label analysis) is run separately.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("scripts").resolve()))

import data_import
import clean_data
import rq1_post_refusal_behavior as rq1
import matching
import contextual as cx

load_dotenv()  # reads LUMEN_API_KEY from .env

WILDCHAT_DIR = Path("data/wildchat")
PROCESSED_DIR = Path("data/processed")

JOINED = PROCESSED_DIR / "refusal_conversations.parquet"
FIRST_REFUSALS = PROCESSED_DIR / "first_refusal_conversations.parquet"
FOLLOWUPS = PROCESSED_DIR / "followup_requests.parquet"
ANNOTATION_SAMPLE = PROCESSED_DIR / "followup_annotation_sample.csv"
PAIRS = PROCESSED_DIR / "request_response_pairs.parquet"
JUDGMENTS = PROCESSED_DIR / "fulfillment_judgments.jsonl"
FULFILLMENT = PROCESSED_DIR / "earliest_fulfillment.parquet"
RAW_LABELS = PROCESSED_DIR / "raw_labels.parquet"
SECOND_REFUSAL = PROCESSED_DIR / "second_refusal.parquet"
STRATEGY_FOLLOWUPS = PROCESSED_DIR / "strategy_followups.parquet"
STRATEGY_EMBEDDINGS = PROCESSED_DIR / "strategy_embeddings.npz"
STRATEGY_SCORES = PROCESSED_DIR / "strategy_scores.parquet"
JUDGE_ANNOTATION_SAMPLE = PROCESSED_DIR / "judge_annotation_sample.csv"
TEST_PAIRS = PROCESSED_DIR / "test_pairs.parquet"
TEST_JUDGMENTS = PROCESSED_DIR / "test_judgments.jsonl"

# NCSA Lumen; the API key is read from LUMEN_API_KEY in .env.
JUDGE_MODEL = "qwen3.8-27b"
JUDGE_URL = "https://lumen.ncsa.illinois.edu/v1"

## 1. Import data
Download WildChat (skip if `data/wildchat` already exists), then join it with RAW refusal labels. The join is skipped if `JOINED` exists, so with a copy of `data/processed` the notebook runs without WildChat access.

In [ ]:
# data_import.download_wildchat(WILDCHAT_DIR)
if JOINED.exists():
    print("Skipping join_with_raw; found", JOINED)
else:
    data_import.join_with_raw(WILDCHAT_DIR, JOINED)

## 2. Clean data

In [ ]:
# Delete an output file to rebuild it (e.g. after re-running the join).
if FIRST_REFUSALS.exists():
    print("Skipping clean_refusals; found", FIRST_REFUSALS)
else:
    clean_data.clean_refusals(JOINED, FIRST_REFUSALS)

if FOLLOWUPS.exists():
    print("Skipping extract_followups; found", FOLLOWUPS)
else:
    clean_data.extract_followups(FIRST_REFUSALS, FOLLOWUPS)

## 3. RQ1: post-refusal behavior

In [ ]:
rq1.continuation_summary(FIRST_REFUSALS)
# Re-sampling would overwrite the manual labels in ANNOTATION_SAMPLE.
if ANNOTATION_SAMPLE.exists():
    print("Skipping sample_followups; found", ANNOTATION_SAMPLE)
else:
    rq1.sample_followups(FOLLOWUPS, ANNOTATION_SAMPLE)

## 4. RQ1.2: Match refused requests to later fulfilling responses
Pair each first refusal with every later assistant response in the window, label each pair with the LLM judge, then keep the earliest full fulfillment per conversation. The judge runs on NCSA Lumen: put your Lumen API key in `.env` (copy `.env.example`). Judging appends to `JUDGMENTS` and skips pairs already labeled, so it can be stopped and resumed.

In [ ]:
if PAIRS.exists():
    print("Skipping build_pairs; found", PAIRS)
else:
    matching.build_pairs(FIRST_REFUSALS, PAIRS)

### Test run (20 conversations)
Judge every pair from 20 random conversations and read the labels before the full run. Results go to separate test files. Delete `TEST_JUDGMENTS` to judge again, e.g. after changing `JUDGE_MODEL`.

In [ ]:
if TEST_PAIRS.exists():
    print("Skipping sample_conversations; found", TEST_PAIRS)
else:
    matching.sample_conversations(PAIRS, TEST_PAIRS, n=20)
matching.judge_pairs(TEST_PAIRS, TEST_JUDGMENTS, model=JUDGE_MODEL, base_url=JUDGE_URL)

In [ ]:
test = matching.load_judgments(TEST_PAIRS, TEST_JUDGMENTS)
print(test["label"].value_counts(dropna=False).to_string())
test[["conversation_id", "candidate_offset", "retry_number", "original_request",
      "candidate_response", "label", "explanation", "evidence"]]

### Full run

In [ ]:
matching.judge_pairs(PAIRS, JUDGMENTS, model=JUDGE_MODEL, base_url=JUDGE_URL)

In [ ]:
matching.earliest_fulfillment(FIRST_REFUSALS, PAIRS, JUDGMENTS, FULFILLMENT)

### Refused again after the first follow-up
Look up RAW's refusal labels for the assistant reply to each conversation's first follow-up, and compare them with the fulfillment judge's label for the same reply. RAW does not label every message, so the rates are among replies with a label.

In [ ]:
if RAW_LABELS.exists():
    print("Skipping save_raw_labels; found", RAW_LABELS)
else:
    data_import.save_raw_labels(FIRST_REFUSALS, RAW_LABELS)

rq1.second_refusal(FIRST_REFUSALS, RAW_LABELS, SECOND_REFUSAL, PAIRS, JUDGMENTS)

### Judge validation
Sample judged pairs, fill in `human_label` (one of `full fulfillment`, `partial fulfillment`, `no fulfillment`, `uncertain`) without looking at `judge_label`, then compare.

In [ ]:
# Re-sampling would overwrite the manual labels in JUDGE_ANNOTATION_SAMPLE.
if JUDGE_ANNOTATION_SAMPLE.exists():
    print("Skipping sample_pairs_for_annotation; found", JUDGE_ANNOTATION_SAMPLE)
else:
    matching.sample_pairs_for_annotation(PAIRS, JUDGMENTS, JUDGE_ANNOTATION_SAMPLE)

In [ ]:
matching.judge_agreement(JUDGE_ANNOTATION_SAMPLE)

## 5. RQ1.3: Contextual analysis
Score four retry strategies on every follow-up `u_k` (k = 1..5) of the refused request `m_i`, then fit one model of whether the reply to `u_k` fully fulfills `m_i`.

| Strategy | Score for `u_k` |
| --- | --- |
| Repetition | cosine of `u_k` and `m_i` (bge-m3 embeddings) |
| Fictional framing | projection of `u_k` minus projection of `m_i` on a fiction direction built from `cx.FICTION_PAIRS` |
| Softening | moderation of `m_i` minus moderation of `u_k` (highest OpenAI category score stored in WildChat); positive = toned down |
| Decomposition | coverage of `m_i`'s clauses by `u_1..u_k` together minus by the best single follow-up; 0 when `m_i` has fewer than 2 clauses |

### 5.1 Follow-up table
One row per follow-up, labeled with the judge's label for the first reply after it. Moderation scores are read from `data/wildchat`; skipped when the table exists.

In [ ]:
if STRATEGY_FOLLOWUPS.exists():
    print("Skipping; found", STRATEGY_FOLLOWUPS)
else:
    cx.build_followup_table(FOLLOWUPS, FIRST_REFUSALS, RAW_LABELS, PAIRS, JUDGMENTS, STRATEGY_FOLLOWUPS)
    cx.add_moderation(WILDCHAT_DIR, STRATEGY_FOLLOWUPS)

### 5.2 Embeddings
Embeds `m_i`, every follow-up, every clause of `m_i`, and the fiction examples with `BAAI/bge-m3` on CPU. Slow; skipped when the file exists.

In [ ]:
if STRATEGY_EMBEDDINGS.exists():
    print("Skipping; found", STRATEGY_EMBEDDINGS)
else:
    cx.embed_texts(STRATEGY_FOLLOWUPS, STRATEGY_EMBEDDINGS)

### 5.3 Strategy scores

In [ ]:
scores = cx.score_strategies(STRATEGY_FOLLOWUPS, STRATEGY_EMBEDDINGS, STRATEGY_SCORES)
display(scores[cx.STRATEGIES].describe())
display(cx.score_correlations(scores).round(2))

### 5.4 Check by eye
Highest- and lowest-scoring follow-ups for each strategy.

In [ ]:
for column in cx.STRATEGIES:
    cx.show_extremes(scores, column)

### 5.5 Model
Linear probability model (OLS) on follow-ups up to each conversation's first full fulfillment. Strategy scores are standardized, so each effect is the change in P(fulfilled) in percentage points per 1 SD. Controls: request moderation, whether `m_i` has 2+ clauses, follow-up number, source model, and pool. Standard errors are clustered by conversation; `p_holm` adjusts for the four strategies.

In [ ]:
main, ols = cx.fit_ols(scores)
print(main.attrs)
display(main.round(3))

### 5.6 Results
Coefficients in percentage points of P(fulfilled), one column per model; `*` marks p < .05. Strategy scores and request moderation are per 1 SD. Model 1 is the main OLS; Model 2 is a logit reported as average marginal effects; Model 3 keeps follow-ups that stay on topic (repetition 0.6 to 0.95); Model 4 keeps refusals that GPT-5.4-mini also labels; Model 5 counts partial fulfillment as success. Reference levels: follow-up 1, GPT-3.5-turbo, general pool.

In [ ]:
cx.results_table(scores)